In [1]:
!pip install fastapi uvicorn sqlalchemy pydantic pytest httpx nest_asyncio

phase 1

In [2]:
import os

# Creating folders
os.makedirs("PFMS/app", exist_ok=True)
os.makedirs("PFMS/tests", exist_ok=True)

# Creating empty files
files = [
    "PFMS/app/main.py",
    "PFMS/app/database.py",
    "PFMS/app/models.py",
    "PFMS/app/schemas.py",
    "PFMS/app/crud.py",
    "PFMS/tests/test_api.py",
    "PFMS/requirements.txt"
]

for file in files:
    open(file, "w").close()

print("PFMS project structure created successfully!")

PFMS project structure created successfully!


In [3]:
# Import SQLAlchemy components required for database configuration
from sqlalchemy import create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

# Configure the SQLite database
DATABASE_URL = "sqlite:///./pfms.db"

# Create the database engine
engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

# Create a session factory for database operations
SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

# Create the base class for database models
Base = declarative_base()

print("SQLite database configuration initialized successfully.")

SQLite database configuration initialized successfully.


In [ ]:
database_code = '''
# Import SQLAlchemy components required for database configuration
from sqlalchemy import create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

# Configure the SQLite database
DATABASE_URL = "sqlite:///./pfms.db"

# Create the database engine
engine = create_engine(
    DATABASE_URL,
    connect_args={"check_same_thread": False}
)

# Create a session factory for database operations
SessionLocal = sessionmaker(
    autocommit=False,
    autoflush=False,
    bind=engine
)

# Create the base class for database models
Base = declarative_base()
'''

with open("PFMS/app/database.py", "w") as file:
    file.write(database_code)

print("database.py created successfully.")

In [8]:
model_code = '''
# Import SQLAlchemy components required for model definition
from sqlalchemy import Column, Integer, Float, DateTime
from datetime import datetime

from .database import Base


class SensorReading(Base):
    # Define the sensor readings table
    __tablename__ = "sensor_readings"

    id = Column(Integer, primary_key=True, index=True)
    field_id = Column(Integer, index=True, nullable=False)
    soil_moisture = Column(Float, nullable=False)
    temperature = Column(Float, nullable=False)
    humidity = Column(Float, nullable=False)
    timestamp = Column(DateTime, default=datetime.utcnow)
'''

with open("PFMS/app/models.py", "w") as file:
    file.write(model_code)

print("models.py created successfully.")

models.py created successfully.


In [9]:
!cat PFMS/app/models.py


# Import SQLAlchemy components required for model definition
from sqlalchemy import Column, Integer, Float, DateTime
from datetime import datetime

from .database import Base


class SensorReading(Base):
    # Define the sensor readings table
    __tablename__ = "sensor_readings"

    id = Column(Integer, primary_key=True, index=True)
    field_id = Column(Integer, index=True, nullable=False)
    soil_moisture = Column(Float, nullable=False)
    temperature = Column(Float, nullable=False)
    humidity = Column(Float, nullable=False)
    timestamp = Column(DateTime, default=datetime.utcnow)


In [10]:
# Import Pydantic components required for data validation
from pydantic import BaseModel, Field
from datetime import datetime


class SensorReadingCreate(BaseModel):
    # Define validation rules for incoming sensor data
    field_id: int
    soil_moisture: float = Field(..., ge=0, le=100)
    temperature: float
    humidity: float = Field(..., ge=0, le=100)


class SensorReadingResponse(BaseModel):
    # Define the structure of sensor data returned by the API
    id: int
    field_id: int
    soil_moisture: float
    temperature: float
    humidity: float
    timestamp: datetime

    class Config:
        from_attributes = True

/tmp/ipykernel_882/576574304.py:14: PydanticDeprecatedSince20: Support for class-based `config` is deprecated, use ConfigDict instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  class SensorReadingResponse(BaseModel):


In [11]:
schema_code = '''
# Import Pydantic components required for data validation
from pydantic import BaseModel, Field
from datetime import datetime


class SensorReadingCreate(BaseModel):
    # Define validation rules for incoming sensor data
    field_id: int
    soil_moisture: float = Field(..., ge=0, le=100)
    temperature: float
    humidity: float = Field(..., ge=0, le=100)


class SensorReadingResponse(BaseModel):
    # Define the structure of sensor data returned by the API
    id: int
    field_id: int
    soil_moisture: float
    temperature: float
    humidity: float
    timestamp: datetime

    class Config:
        from_attributes = True
'''

with open("PFMS/app/schemas.py", "w") as file:
    file.write(schema_code)

print("schemas.py created successfully.")

schemas.py created successfully.


In [12]:
!cat PFMS/app/schemas.py


# Import Pydantic components required for data validation
from pydantic import BaseModel, Field
from datetime import datetime


class SensorReadingCreate(BaseModel):
    # Define validation rules for incoming sensor data
    field_id: int
    soil_moisture: float = Field(..., ge=0, le=100)
    temperature: float
    humidity: float = Field(..., ge=0, le=100)


class SensorReadingResponse(BaseModel):
    # Define the structure of sensor data returned by the API
    id: int
    field_id: int
    soil_moisture: float
    temperature: float
    humidity: float
    timestamp: datetime

    class Config:
        from_attributes = True


In [14]:
# Create CRUD operations for sensor readings

crud_code = '''
# Import SQLAlchemy session and database models
from sqlalchemy.orm import Session

from . import models, schemas


def create_sensor_reading(db: Session, reading: schemas.SensorReadingCreate):
    # Create a new sensor reading
    db_reading = models.SensorReading(**reading.model_dump())

    db.add(db_reading)
    db.commit()
    db.refresh(db_reading)

    return db_reading


def get_sensor_readings(db: Session):
    # Retrieve all sensor readings
    return db.query(models.SensorReading).all()


def get_sensor_reading(db: Session, reading_id: int):
    # Retrieve a sensor reading using its ID
    return db.query(models.SensorReading).filter(
        models.SensorReading.id == reading_id
    ).first()


def update_sensor_reading(
    db: Session,
    reading_id: int,
    reading: schemas.SensorReadingCreate
):
    # Update an existing sensor reading
    db_reading = get_sensor_reading(db, reading_id)

    if db_reading is None:
        return None

    for key, value in reading.model_dump().items():
        setattr(db_reading, key, value)

    db.commit()
    db.refresh(db_reading)

    return db_reading


def delete_sensor_reading(db: Session, reading_id: int):
    # Delete an existing sensor reading
    db_reading = get_sensor_reading(db, reading_id)

    if db_reading is None:
        return None

    db.delete(db_reading)
    db.commit()

    return db_reading
'''

# Save the CRUD operations to the project file
with open("PFMS/app/crud.py", "w") as file:
    file.write(crud_code)

print("crud.py created successfully.")

crud.py created successfully.


In [15]:
!cat PFMS/app/crud.py


# Import SQLAlchemy session and database models
from sqlalchemy.orm import Session

from . import models, schemas


def create_sensor_reading(db: Session, reading: schemas.SensorReadingCreate):
    # Create a new sensor reading
    db_reading = models.SensorReading(**reading.model_dump())

    db.add(db_reading)
    db.commit()
    db.refresh(db_reading)

    return db_reading


def get_sensor_readings(db: Session):
    # Retrieve all sensor readings
    return db.query(models.SensorReading).all()


def get_sensor_reading(db: Session, reading_id: int):
    # Retrieve a sensor reading using its ID
    return db.query(models.SensorReading).filter(
        models.SensorReading.id == reading_id
    ).first()


def update_sensor_reading(
    db: Session,
    reading_id: int,
    reading: schemas.SensorReadingCreate
):
    # Update an existing sensor reading
    db_reading = get_sensor_reading(db, reading_id)

    if db_reading is None:
        return None

    for key, value in reading.mod

In [16]:
# Create the main FastAPI application

main_code = '''
# Import FastAPI components and database dependencies
from fastapi import FastAPI, Depends, HTTPException
from sqlalchemy.orm import Session

from .database import SessionLocal, engine, Base
from . import models, schemas, crud

# Create database tables
Base.metadata.create_all(bind=engine)

# Initialize the FastAPI application
app = FastAPI(
    title="Precision Farming Management System",
    description="REST API for Agricultural Monitoring and Resource Management",
    version="1.0.0"
)


# Provide a database session for each request
def get_db():
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()


@app.get("/")
def root():
    # Return the API status
    return {"message": "Precision Farming Management System API is running"}


@app.post("/sensor-readings", response_model=schemas.SensorReadingResponse)
def create_sensor_reading(
    reading: schemas.SensorReadingCreate,
    db: Session = Depends(get_db)
):
    # Create a new sensor reading
    return crud.create_sensor_reading(db, reading)


@app.get("/sensor-readings", response_model=list[schemas.SensorReadingResponse])
def get_sensor_readings(db: Session = Depends(get_db)):
    # Retrieve all sensor readings
    return crud.get_sensor_readings(db)


@app.get("/sensor-readings/{reading_id}", response_model=schemas.SensorReadingResponse)
def get_sensor_reading(
    reading_id: int,
    db: Session = Depends(get_db)
):
    # Retrieve a sensor reading by ID
    reading = crud.get_sensor_reading(db, reading_id)

    if reading is None:
        raise HTTPException(status_code=404, detail="Sensor reading not found")

    return reading


@app.put("/sensor-readings/{reading_id}", response_model=schemas.SensorReadingResponse)
def update_sensor_reading(
    reading_id: int,
    reading: schemas.SensorReadingCreate,
    db: Session = Depends(get_db)
):
    # Update an existing sensor reading
    updated_reading = crud.update_sensor_reading(
        db,
        reading_id,
        reading
    )

    if updated_reading is None:
        raise HTTPException(status_code=404, detail="Sensor reading not found")

    return updated_reading


@app.delete("/sensor-readings/{reading_id}")
def delete_sensor_reading(
    reading_id: int,
    db: Session = Depends(get_db)
):
    # Delete an existing sensor reading
    deleted_reading = crud.delete_sensor_reading(db, reading_id)

    if deleted_reading is None:
        raise HTTPException(status_code=404, detail="Sensor reading not found")

    return {
        "message": "Sensor reading deleted successfully",
        "id": reading_id
    }
'''

# Save the FastAPI application
with open("PFMS/app/main.py", "w") as file:
    file.write(main_code)

print("main.py created successfully.")

main.py created successfully.


In [17]:
# Initialize the application and test packages

open("PFMS/app/__init__.py", "w").close()
open("PFMS/tests/__init__.py", "w").close()

print("Project packages initialized successfully.")

Project packages initialized successfully.


In [18]:
!find PFMS -type f

PFMS/tests/__init__.py
PFMS/tests/test_api.py
PFMS/requirements.txt
PFMS/app/__init__.py
PFMS/app/models.py
PFMS/app/schemas.py
PFMS/app/main.py
PFMS/app/crud.py
PFMS/app/database.py


In [19]:
# Start the FastAPI development server

import subprocess
import time

server = subprocess.Popen(
    [
        "uvicorn",
        "PFMS.app.main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ]
)

time.sleep(3)

print("FastAPI server started successfully on port 8000.")

FastAPI server started successfully on port 8000.


In [23]:
# Start the FastAPI development server in the background

import subprocess
import time
import requests

server = subprocess.Popen(
    [
        "uvicorn",
        "PFMS.app.main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ]
)

# Give the server a moment to start
time.sleep(3)

# Verify that the API is responding
response = requests.get("http://127.0.0.1:8000/")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'message': 'Precision Farming Management System API is running'}


In [24]:
# Open the FastAPI Swagger documentation through the Colab proxy

from google.colab import output

output.serve_kernel_port_as_iframe(8000)

<IPython.core.display.Javascript object>

In [27]:
# Generate the Colab proxy URL for the FastAPI Swagger documentation

from google.colab import output

docs_url = output.eval_js(
    "(async () => { const baseUrl = await google.colab.kernel.proxyPort(8000); return baseUrl + '/docs'; })()"
)

print("Swagger UI:")
print(docs_url)

Swagger UI:
[object Promise]/docs


In [28]:
# Verify that FastAPI's Swagger documentation endpoint is available

import requests

response = requests.get("http://127.0.0.1:8000/docs")

print("Status code:", response.status_code)
print("Swagger UI available:", response.status_code == 200)

Status code: 200
Swagger UI available: True


In [29]:
# Test the sensor reading creation endpoint

import requests

sensor_data = {
    "field_id": 1,
    "soil_moisture": 42.5,
    "temperature": 29.3,
    "humidity": 68.0
}

response = requests.post(
    "http://127.0.0.1:8000/sensor-readings",
    json=sensor_data
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'id': 1, 'field_id': 1, 'soil_moisture': 42.5, 'temperature': 29.3, 'humidity': 68.0, 'timestamp': '2026-09-26T15:50:10.023410'}


In [30]:
# Test the endpoint that retrieves all sensor readings

import requests

response = requests.get(
    "http://127.0.0.1:8000/sensor-readings"
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: [{'id': 1, 'field_id': 1, 'soil_moisture': 42.5, 'temperature': 29.3, 'humidity': 68.0, 'timestamp': '2026-09-26T15:50:10.023410'}]


In [31]:
# Test the endpoint that retrieves a specific sensor reading

import requests

reading_id = 1

response = requests.get(
    f"http://127.0.0.1:8000/sensor-readings/{reading_id}"
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'id': 1, 'field_id': 1, 'soil_moisture': 42.5, 'temperature': 29.3, 'humidity': 68.0, 'timestamp': '2026-09-26T15:50:10.023410'}


In [32]:
# Test the endpoint that updates an existing sensor reading

import requests

reading_id = 1

updated_sensor_data = {
    "field_id": 1,
    "soil_moisture": 55.0,
    "temperature": 30.1,
    "humidity": 72.0
}

response = requests.put(
    f"http://127.0.0.1:8000/sensor-readings/{reading_id}",
    json=updated_sensor_data
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'id': 1, 'field_id': 1, 'soil_moisture': 55.0, 'temperature': 30.1, 'humidity': 72.0, 'timestamp': '2026-09-26T15:50:10.023410'}


In [33]:
# Test the endpoint that deletes an existing sensor reading

import requests

reading_id = 1

response = requests.delete(
    f"http://127.0.0.1:8000/sensor-readings/{reading_id}"
)

print("Status code:", response.status_code)
print("Response:", response.json())


Status code: 200
Response: {'message': 'Sensor reading deleted successfully', 'id': 1}


In [34]:
# Test API validation using an invalid soil moisture value

import requests

invalid_sensor_data = {
    "field_id": 1,
    "soil_moisture": 150,
    "temperature": 30.0,
    "humidity": 70.0
}

response = requests.post(
    "http://127.0.0.1:8000/sensor-readings",
    json=invalid_sensor_data
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 422
Response: {'detail': [{'type': 'less_than_equal', 'loc': ['body', 'soil_moisture'], 'msg': 'Input should be less than or equal to 100', 'input': 150, 'ctx': {'le': 100.0}}]}


In [35]:
# Test the lower boundary value for soil moisture

boundary_sensor_data = {
    "field_id": 1,
    "soil_moisture": 0,
    "temperature": 30.0,
    "humidity": 70.0
}

response = requests.post(
    "http://127.0.0.1:8000/sensor-readings",
    json=boundary_sensor_data
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'id': 1, 'field_id': 1, 'soil_moisture': 0.0, 'temperature': 30.0, 'humidity': 70.0, 'timestamp': '2026-09-26T15:53:44.009133'}


In [36]:
# Test the upper boundary value for soil moisture

boundary_sensor_data = {
    "field_id": 1,
    "soil_moisture": 100,
    "temperature": 30.0,
    "humidity": 70.0
}

response = requests.post(
    "http://127.0.0.1:8000/sensor-readings",
    json=boundary_sensor_data
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'id': 2, 'field_id': 1, 'soil_moisture': 100.0, 'temperature': 30.0, 'humidity': 70.0, 'timestamp': '2026-09-26T15:54:17.178727'}


In [37]:
# Test a value just above the maximum allowed soil moisture

invalid_boundary_data = {
    "field_id": 1,
    "soil_moisture": 100.1,
    "temperature": 30.0,
    "humidity": 70.0
}

response = requests.post(
    "http://127.0.0.1:8000/sensor-readings",
    json=invalid_boundary_data
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 422
Response: {'detail': [{'type': 'less_than_equal', 'loc': ['body', 'soil_moisture'], 'msg': 'Input should be less than or equal to 100', 'input': 100.1, 'ctx': {'le': 100.0}}]}


In [38]:
# Create automated tests for the PFMS REST API

test_code = '''
# Import FastAPI testing utilities
from fastapi.testclient import TestClient

from PFMS.app.main import app


# Create a test client for sending requests to the API
client = TestClient(app)


def test_root_endpoint():
    # Verify that the API is running successfully
    response = client.get("/")

    assert response.status_code == 200
    assert response.json()["message"] == (
        "Precision Farming Management System API is running"
    )


def test_create_sensor_reading():
    # Verify that valid sensor data can be created
    sensor_data = {
        "field_id": 1,
        "soil_moisture": 45.5,
        "temperature": 29.0,
        "humidity": 65.0
    }

    response = client.post(
        "/sensor-readings",
        json=sensor_data
    )

    assert response.status_code == 200

    data = response.json()

    assert data["field_id"] == 1
    assert data["soil_moisture"] == 45.5
    assert data["temperature"] == 29.0
    assert data["humidity"] == 65.0


def test_invalid_soil_moisture():
    # Verify that soil moisture above 100% is rejected
    sensor_data = {
        "field_id": 1,
        "soil_moisture": 150,
        "temperature": 30.0,
        "humidity": 70.0
    }

    response = client.post(
        "/sensor-readings",
        json=sensor_data
    )

    assert response.status_code == 422


def test_get_sensor_readings():
    # Verify that sensor readings can be retrieved
    response = client.get("/sensor-readings")

    assert response.status_code == 200
    assert isinstance(response.json(), list)


def test_get_nonexistent_sensor_reading():
    # Verify that requesting a missing reading returns 404
    response = client.get("/sensor-readings/99999")

    assert response.status_code == 404
'''

with open("PFMS/tests/test_api.py", "w") as file:
    file.write(test_code)

print("Pytest test suite created successfully.")

Pytest test suite created successfully.


In [40]:
# Make sure the PFMS directories are recognized as Python packages

open("PFMS/__init__.py", "w").close()
open("PFMS/app/__init__.py", "w").close()
open("PFMS/tests/__init__.py", "w").close()

# Add the project root to Python's module search path
import sys

project_root = "/content"
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print("PFMS package configuration updated successfully.")
print("Project root:", project_root)

PFMS package configuration updated successfully.
Project root: /content


In [41]:
# Run the PFMS automated API test suite using pytest

!pytest PFMS/tests/test_api.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.12.1, anyio-4.14.2, typeguard-4.6.0
collected 5 items                                                              

PFMS/tests/test_api.py::test_root_endpoint PASSED                        [ 20%]
PFMS/tests/test_api.py::test_create_sensor_reading PASSED                [ 40%]
PFMS/tests/test_api.py::test_invalid_soil_moisture PASSED                [ 60%]
PFMS/tests/test_api.py::test_get_sensor_readings PASSED                  [ 80%]
PFMS/tests/test_api.py::test_get_nonexistent_sensor_reading PASSED       [100%]

============================== 5 passed in 0.61s ===============================


In [42]:
# Add boundary value tests to the automated test suite

boundary_tests = '''

def test_soil_moisture_lower_boundary():
    # Verify that the minimum valid soil moisture value is accepted
    sensor_data = {
        "field_id": 1,
        "soil_moisture": 0,
        "temperature": 30.0,
        "humidity": 70.0
    }

    response = client.post(
        "/sensor-readings",
        json=sensor_data
    )

    assert response.status_code == 200


def test_soil_moisture_upper_boundary():
    # Verify that the maximum valid soil moisture value is accepted
    sensor_data = {
        "field_id": 1,
        "soil_moisture": 100,
        "temperature": 30.0,
        "humidity": 70.0
    }

    response = client.post(
        "/sensor-readings",
        json=sensor_data
    )

    assert response.status_code == 200


def test_soil_moisture_above_upper_boundary():
    # Verify that a value just above the maximum is rejected
    sensor_data = {
        "field_id": 1,
        "soil_moisture": 100.1,
        "temperature": 30.0,
        "humidity": 70.0
    }

    response = client.post(
        "/sensor-readings",
        json=sensor_data
    )

    assert response.status_code == 422
'''

with open("PFMS/tests/test_api.py", "a") as file:
    file.write(boundary_tests)

print("Boundary value tests added successfully.")

Boundary value tests added successfully.


In [43]:
# Run the complete PFMS automated test suite

!pytest PFMS/tests/test_api.py -v

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.12.1, anyio-4.14.2, typeguard-4.6.0
collected 8 items                                                              

PFMS/tests/test_api.py::test_root_endpoint PASSED                        [ 12%]
PFMS/tests/test_api.py::test_create_sensor_reading PASSED                [ 25%]
PFMS/tests/test_api.py::test_invalid_soil_moisture PASSED                [ 37%]
PFMS/tests/test_api.py::test_get_sensor_readings PASSED                  [ 50%]
PFMS/tests/test_api.py::test_get_nonexistent_sensor_reading PASSED       [ 62%]
PFMS/tests/test_api.py::test_soil_moisture_lower_boundary PASSED         [ 75%]
PFMS/tests/test_api.py::test_soil_moisture_upper_boundary PASSED         [ 87%]
PFMS/tests/test_api.py::test_soil_moisture_above_upper_boundary PASSED   [100%]

=================

In [44]:
# Create the irrigation recommendation service

irrigation_code = '''
def get_irrigation_recommendation(soil_moisture: float):
    # Determine the irrigation recommendation based on soil moisture
    if soil_moisture < 30:
        return {
            "soil_moisture": soil_moisture,
            "irrigation_required": True,
            "recommendation": "Irrigation required"
        }

    if soil_moisture <= 60:
        return {
            "soil_moisture": soil_moisture,
            "irrigation_required": False,
            "recommendation": "Monitor soil moisture"
        }

    return {
        "soil_moisture": soil_moisture,
        "irrigation_required": False,
        "recommendation": "No irrigation required"
    }
'''

with open("PFMS/app/irrigation.py", "w") as file:
    file.write(irrigation_code)

print("Irrigation recommendation service created successfully.")

Irrigation recommendation service created successfully.


In [45]:
# Add the irrigation recommendation endpoint to the FastAPI application

main_path = "PFMS/app/main.py"

with open(main_path, "r") as file:
    main_code = file.read()

# Import the irrigation recommendation service
if "from .irrigation import get_irrigation_recommendation" not in main_code:
    main_code = main_code.replace(
        "from . import models, schemas, crud",
        "from . import models, schemas, crud\nfrom .irrigation import get_irrigation_recommendation"
    )

# Add the irrigation endpoint
irrigation_endpoint = '''

@app.get("/irrigation-recommendation/{soil_moisture}")
def irrigation_recommendation(soil_moisture: float):
    # Return an irrigation recommendation based on soil moisture
    return get_irrigation_recommendation(soil_moisture)
'''

if '"/irrigation-recommendation/{soil_moisture}"' not in main_code:
    main_code += irrigation_endpoint

with open(main_path, "w") as file:
    file.write(main_code)

print("Irrigation recommendation endpoint added successfully.")

Irrigation recommendation endpoint added successfully.


In [46]:
# Restart the FastAPI server so the new irrigation endpoint is loaded

server.terminate()
server.wait()

import subprocess
import time
import requests

server = subprocess.Popen(
    [
        "uvicorn",
        "PFMS.app.main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ]
)

time.sleep(3)

response = requests.get("http://127.0.0.1:8000/")

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'message': 'Precision Farming Management System API is running'}


In [47]:
# Test the irrigation recommendation for dry soil conditions

import requests

soil_moisture = 25

response = requests.get(
    f"http://127.0.0.1:8000/irrigation-recommendation/{soil_moisture}"
)

print("Status code:", response.status_code)
print("Response:", response.json())

Status code: 200
Response: {'soil_moisture': 25.0, 'irrigation_required': True, 'recommendation': 'Irrigation required'}


Phase 2 is an extension of that same application:

Phase 1 + fields + simulated data + irrigation logic + inventory + better validation + more tests

In [49]:
# ============================================================
# PFMS - Phase 2
# Extending the basic sensor API with farming functionality
# ============================================================

import subprocess
import time
import requests


# ------------------------------------------------------------
# Update the database models
# ------------------------------------------------------------

models_code = '''
from sqlalchemy import Column, Integer, Float, String, DateTime
from datetime import datetime

from .database import Base


class Field(Base):
    # Basic information about each field managed by the system.
    __tablename__ = "fields"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, nullable=False)
    location = Column(String, nullable=False)
    crop_type = Column(String, nullable=False)


class SensorReading(Base):
    # Stores environmental readings received from field sensors.
    __tablename__ = "sensor_readings"

    id = Column(Integer, primary_key=True, index=True)
    field_id = Column(Integer, index=True, nullable=False)
    soil_moisture = Column(Float, nullable=False)
    temperature = Column(Float, nullable=False)
    humidity = Column(Float, nullable=False)
    timestamp = Column(DateTime, default=datetime.utcnow)


class InventoryItem(Base):
    # Tracks fertilizers and other agricultural resources.
    __tablename__ = "inventory_items"

    id = Column(Integer, primary_key=True, index=True)
    name = Column(String, nullable=False)
    category = Column(String, nullable=False)
    quantity = Column(Float, nullable=False)
    unit = Column(String, nullable=False)
'''

with open("PFMS/app/models.py", "w") as file:
    file.write(models_code)


# ------------------------------------------------------------
# API schemas and validation rules
# ------------------------------------------------------------

schemas_code = '''
from pydantic import BaseModel, Field
from datetime import datetime


class FieldCreate(BaseModel):
    name: str = Field(..., min_length=2, max_length=100)
    location: str = Field(..., min_length=2, max_length=150)
    crop_type: str = Field(..., min_length=2, max_length=100)


class FieldResponse(BaseModel):
    id: int
    name: str
    location: str
    crop_type: str

    class Config:
        from_attributes = True


class SensorReadingCreate(BaseModel):
    # Sensor values are restricted to realistic ranges.
    field_id: int = Field(..., gt=0)
    soil_moisture: float = Field(..., ge=0, le=100)
    temperature: float = Field(..., ge=-50, le=80)
    humidity: float = Field(..., ge=0, le=100)


class SensorReadingResponse(BaseModel):
    id: int
    field_id: int
    soil_moisture: float
    temperature: float
    humidity: float
    timestamp: datetime

    class Config:
        from_attributes = True


class InventoryItemCreate(BaseModel):
    name: str = Field(..., min_length=2, max_length=100)
    category: str = Field(..., min_length=2, max_length=50)
    quantity: float = Field(..., ge=0)
    unit: str = Field(..., min_length=1, max_length=20)


class InventoryItemResponse(BaseModel):
    id: int
    name: str
    category: str
    quantity: float
    unit: str

    class Config:
        from_attributes = True
'''

with open("PFMS/app/schemas.py", "w") as file:
    file.write(schemas_code)


# ------------------------------------------------------------
# Irrigation decision logic
# ------------------------------------------------------------

irrigation_code = '''
def get_irrigation_recommendation(soil_moisture: float):
    # These thresholds provide a simple first version of the
    # irrigation decision logic. They can be replaced later
    # with crop-specific or ML-based recommendations.

    if soil_moisture < 30:
        return {
            "soil_moisture": soil_moisture,
            "irrigation_required": True,
            "recommendation": "Irrigation required"
        }

    if soil_moisture <= 60:
        return {
            "soil_moisture": soil_moisture,
            "irrigation_required": False,
            "recommendation": "Monitor soil moisture"
        }

    return {
        "soil_moisture": soil_moisture,
        "irrigation_required": False,
        "recommendation": "No irrigation required"
    }
'''

with open("PFMS/app/irrigation.py", "w") as file:
    file.write(irrigation_code)


# ------------------------------------------------------------
# Database operations
# ------------------------------------------------------------

crud_code = '''
from sqlalchemy.orm import Session

from . import models, schemas


# -------------------- Fields --------------------

def create_field(db: Session, field: schemas.FieldCreate):
    db_field = models.Field(**field.model_dump())

    db.add(db_field)
    db.commit()
    db.refresh(db_field)

    return db_field


def get_fields(db: Session):
    return db.query(models.Field).all()


def get_field(db: Session, field_id: int):
    return db.query(models.Field).filter(
        models.Field.id == field_id
    ).first()


# -------------------- Sensor readings --------------------

def create_sensor_reading(db: Session, reading: schemas.SensorReadingCreate):
    db_reading = models.SensorReading(**reading.model_dump())

    db.add(db_reading)
    db.commit()
    db.refresh(db_reading)

    return db_reading


def get_sensor_readings(db: Session):
    return db.query(models.SensorReading).all()


def get_sensor_reading(db: Session, reading_id: int):
    return db.query(models.SensorReading).filter(
        models.SensorReading.id == reading_id
    ).first()


def update_sensor_reading(
    db: Session,
    reading_id: int,
    reading: schemas.SensorReadingCreate
):
    db_reading = get_sensor_reading(db, reading_id)

    if db_reading is None:
        return None

    for key, value in reading.model_dump().items():
        setattr(db_reading, key, value)

    db.commit()
    db.refresh(db_reading)

    return db_reading


def delete_sensor_reading(db: Session, reading_id: int):
    db_reading = get_sensor_reading(db, reading_id)

    if db_reading is None:
        return None

    db.delete(db_reading)
    db.commit()

    return db_reading


# -------------------- Inventory --------------------

def create_inventory_item(
    db: Session,
    item: schemas.InventoryItemCreate
):
    db_item = models.InventoryItem(**item.model_dump())

    db.add(db_item)
    db.commit()
    db.refresh(db_item)

    return db_item


def get_inventory_items(db: Session):
    return db.query(models.InventoryItem).all()
'''

with open("PFMS/app/crud.py", "w") as file:
    file.write(crud_code)


# ------------------------------------------------------------
# FastAPI routes
# ------------------------------------------------------------

main_code = '''
from fastapi import FastAPI, Depends, HTTPException
from sqlalchemy.orm import Session

from .database import SessionLocal, engine, Base
from . import models, schemas, crud
from .irrigation import get_irrigation_recommendation


Base.metadata.create_all(bind=engine)


app = FastAPI(
    title="Precision Farming Management System",
    description="REST API for Agricultural Monitoring and Resource Management",
    version="1.0.0"
)


def get_db():
    db = SessionLocal()

    try:
        yield db
    finally:
        db.close()


@app.get("/")
def root():
    return {
        "message": "Precision Farming Management System API is running"
    }


# ============================================================
# Field management
# ============================================================

@app.post(
    "/fields",
    response_model=schemas.FieldResponse,
    status_code=201
)
def create_field(
    field: schemas.FieldCreate,
    db: Session = Depends(get_db)
):
    return crud.create_field(db, field)


@app.get(
    "/fields",
    response_model=list[schemas.FieldResponse]
)
def get_fields(db: Session = Depends(get_db)):
    return crud.get_fields(db)


@app.get(
    "/fields/{field_id}",
    response_model=schemas.FieldResponse
)
def get_field(
    field_id: int,
    db: Session = Depends(get_db)
):
    field = crud.get_field(db, field_id)

    if field is None:
        raise HTTPException(
            status_code=404,
            detail="Field not found"
        )

    return field


# ============================================================
# Sensor readings
# ============================================================

@app.post(
    "/sensor-readings",
    response_model=schemas.SensorReadingResponse
)
def create_sensor_reading(
    reading: schemas.SensorReadingCreate,
    db: Session = Depends(get_db)
):
    # A reading should always belong to a registered field.
    if crud.get_field(db, reading.field_id) is None:
        raise HTTPException(
            status_code=404,
            detail="Field not found"
        )

    return crud.create_sensor_reading(db, reading)


@app.get(
    "/sensor-readings",
    response_model=list[schemas.SensorReadingResponse]
)
def get_sensor_readings(db: Session = Depends(get_db)):
    return crud.get_sensor_readings(db)


@app.get(
    "/sensor-readings/{reading_id}",
    response_model=schemas.SensorReadingResponse
)
def get_sensor_reading(
    reading_id: int,
    db: Session = Depends(get_db)
):
    reading = crud.get_sensor_reading(db, reading_id)

    if reading is None:
        raise HTTPException(
            status_code=404,
            detail="Sensor reading not found"
        )

    return reading


@app.put(
    "/sensor-readings/{reading_id}",
    response_model=schemas.SensorReadingResponse
)
def update_sensor_reading(
    reading_id: int,
    reading: schemas.SensorReadingCreate,
    db: Session = Depends(get_db)
):
    if crud.get_field(db, reading.field_id) is None:
        raise HTTPException(
            status_code=404,
            detail="Field not found"
        )

    updated_reading = crud.update_sensor_reading(
        db,
        reading_id,
        reading
    )

    if updated_reading is None:
        raise HTTPException(
            status_code=404,
            detail="Sensor reading not found"
        )

    return updated_reading


@app.delete("/sensor-readings/{reading_id}")
def delete_sensor_reading(
    reading_id: int,
    db: Session = Depends(get_db)
):
    deleted_reading = crud.delete_sensor_reading(
        db,
        reading_id
    )

    if deleted_reading is None:
        raise HTTPException(
            status_code=404,
            detail="Sensor reading not found"
        )

    return {
        "message": "Sensor reading deleted successfully",
        "id": reading_id
    }


# ============================================================
# Irrigation recommendations
# ============================================================

@app.get("/irrigation-recommendation/{soil_moisture}")
def irrigation_recommendation(soil_moisture: float):
    if soil_moisture < 0 or soil_moisture > 100:
        raise HTTPException(
            status_code=400,
            detail="Soil moisture must be between 0 and 100"
        )

    return get_irrigation_recommendation(soil_moisture)


@app.get(
    "/sensor-readings/{reading_id}/irrigation-recommendation"
)
def sensor_irrigation_recommendation(
    reading_id: int,
    db: Session = Depends(get_db)
):
    # This endpoint connects an actual stored reading to
    # the irrigation decision instead of requiring the
    # client to provide the moisture value separately.
    reading = crud.get_sensor_reading(db, reading_id)

    if reading is None:
        raise HTTPException(
            status_code=404,
            detail="Sensor reading not found"
        )

    recommendation = get_irrigation_recommendation(
        reading.soil_moisture
    )

    return {
        "reading_id": reading.id,
        "field_id": reading.field_id,
        **recommendation
    }


# ============================================================
# Fertilizer and resource inventory
# ============================================================

@app.post(
    "/inventory",
    response_model=schemas.InventoryItemResponse,
    status_code=201
)
def create_inventory_item(
    item: schemas.InventoryItemCreate,
    db: Session = Depends(get_db)
):
    return crud.create_inventory_item(db, item)


@app.get(
    "/inventory",
    response_model=list[schemas.InventoryItemResponse]
)
def get_inventory_items(db: Session = Depends(get_db)):
    return crud.get_inventory_items(db)
'''

with open("PFMS/app/main.py", "w") as file:
    file.write(main_code)


# ------------------------------------------------------------
# Generate simulated data
# ------------------------------------------------------------

seed_code = '''
from PFMS.app.database import SessionLocal, Base, engine
from PFMS.app import models


Base.metadata.create_all(bind=engine)

db = SessionLocal()

try:
    fields = [
        {
            "name": "North Field",
            "location": "Chennai",
            "crop_type": "Tomato"
        },
        {
            "name": "East Field",
            "location": "Coimbatore",
            "crop_type": "Chilli"
        },
        {
            "name": "Greenhouse Field",
            "location": "Bengaluru",
            "crop_type": "Cucumber"
        }
    ]

    field_objects = []

    for field_data in fields:
        existing = db.query(models.Field).filter(
            models.Field.name == field_data["name"]
        ).first()

        if existing:
            field_objects.append(existing)
        else:
            field = models.Field(**field_data)
            db.add(field)
            db.flush()
            field_objects.append(field)

    # These values simulate what sensors might report from
    # fields with different environmental conditions.
    if db.query(models.SensorReading).count() < 5:
        readings = [
            {
                "field_id": field_objects[0].id,
                "soil_moisture": 24.5,
                "temperature": 31.2,
                "humidity": 64.0
            },
            {
                "field_id": field_objects[0].id,
                "soil_moisture": 47.0,
                "temperature": 29.8,
                "humidity": 69.0
            },
            {
                "field_id": field_objects[1].id,
                "soil_moisture": 28.0,
                "temperature": 30.5,
                "humidity": 61.0
            },
            {
                "field_id": field_objects[1].id,
                "soil_moisture": 72.0,
                "temperature": 28.4,
                "humidity": 74.0
            },
            {
                "field_id": field_objects[2].id,
                "soil_moisture": 55.0,
                "temperature": 27.9,
                "humidity": 78.0
            }
        ]

        for reading_data in readings:
            db.add(models.SensorReading(**reading_data))

    inventory = [
        {
            "name": "NPK 10-26-26",
            "category": "Fertilizer",
            "quantity": 50.0,
            "unit": "kg"
        },
        {
            "name": "Urea",
            "category": "Nitrogen Fertilizer",
            "quantity": 30.0,
            "unit": "kg"
        },
        {
            "name": "Potassium Sulphate",
            "category": "Potassium Fertilizer",
            "quantity": 20.0,
            "unit": "kg"
        }
    ]

    if db.query(models.InventoryItem).count() == 0:
        for item_data in inventory:
            db.add(models.InventoryItem(**item_data))

    db.commit()

    print("Sample PFMS data added successfully.")

finally:
    db.close()
'''

with open("PFMS/seed_data.py", "w") as file:
    file.write(seed_code)


# ------------------------------------------------------------
# Restart the API
# ------------------------------------------------------------

try:
    server.terminate()
    server.wait(timeout=5)
except Exception:
    pass

server = subprocess.Popen(
    [
        "uvicorn",
        "PFMS.app.main:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000"
    ]
)

time.sleep(3)


# --- Add PYTHONPATH setup here ---
import os
import sys

# Get the current PYTHONPATH or initialize it if not set
python_path = os.environ.get('PYTHONPATH', '')

# Add the project root to PYTHONPATH for the subprocess
# We need to add the parent directory of PFMS to PYTHONPATH
project_root = "/content" # Assuming /content is where PFMS is located

if project_root not in python_path.split(os.pathsep):
    # Ensure project_root is at the beginning of PYTHONPATH
    # os.pathsep is ':' on Unix-like systems
    python_path = project_root + os.pathsep + python_path if python_path else project_root

env_vars = os.environ.copy()
env_vars['PYTHONPATH'] = python_path
# --- End PYTHONPATH setup ---


# ------------------------------------------------------------
# Add the simulated data
# ------------------------------------------------------------

seed_result = subprocess.run(
    ["python", "PFMS/seed_data.py"],
    capture_output=True,
    text=True,
    env=env_vars # Pass the modified environment variables
)

print(seed_result.stdout)

if seed_result.stderr:
    print(seed_result.stderr)


# ------------------------------------------------------------
# Quick API check
# ------------------------------------------------------------

response = requests.get("http://127.0.0.1:8000/")

print("API status:", response.status_code)
print("API response:", response.json())


# ------------------------------------------------------------
# Phase 2 automated tests
# ------------------------------------------------------------

test_code = '''
from fastapi.testclient import TestClient

from PFMS.app.main import app


client = TestClient(app)


def test_root_endpoint():
    response = client.get("/")

    assert response.status_code == 200
    assert "Precision Farming" in response.json()["message"]


def test_create_field():
    field = {
        "name": "Test Field",
        "location": "Chennai",
        "crop_type": "Rice"
    }

    response = client.post("/fields", json=field)

    assert response.status_code == 201
    assert response.json()["name"] == "Test Field"


def test_get_fields():
    response = client.get("/fields")

    assert response.status_code == 200
    assert isinstance(response.json(), list)
    assert len(response.json()) >= 1


def test_get_nonexistent_field():
    response = client.get("/fields/99999")

    assert response.status_code == 404


def test_create_sensor_reading():
    fields_response = client.get("/fields")
    field_id = fields_response.json()[0]["id"]

    sensor_data = {
        "field_id": field_id,
        "soil_moisture": 45.5,
        "temperature": 29.0,
        "humidity": 65.0
    }

    response = client.post(
        "/sensor-readings",
        json=sensor_data
    )

    assert response.status_code == 200
    assert response.json()["soil_moisture"] == 45.5


def test_sensor_reading_invalid_field():
    sensor_data = {
        "field_id": 99999,
        "soil_moisture": 45.0,
        "temperature": 29.0,
        "humidity": 65.0
    }

    response = client.post(
        "/sensor-readings",
        json=sensor_data
    )

    assert response.status_code == 404


def test_invalid_soil_moisture():
    sensor_data = {
        "field_id": 1,
        "soil_moisture": 150,
        "temperature": 30.0,
        "humidity": 70.0
    }

    response = client.post(
        "/sensor-readings",
        json=sensor_data
    )

    assert response.status_code == 422


def test_get_sensor_readings():
    response = client.get("/sensor-readings")

    assert response.status_code == 200
    assert isinstance(response.json(), list)


def test_get_nonexistent_sensor_reading():
    response = client.get("/sensor-readings/99999")

    assert response.status_code == 404


def test_soil_moisture_lower_boundary():
    field_id = client.get("/fields").json()[0]["id"]

    sensor_data = {
        "field_id": field_id,
        "soil_moisture": 0,
        "temperature": 30.0,
        "humidity": 70.0
    }

    response = client.post(
        "/sensor-readings",
        json=sensor_data
    )

    assert response.status_code == 200


def test_soil_moisture_upper_boundary():
    field_id = client.get("/fields").json()[0]["id"]

    sensor_data = {
        "field_id": field_id,
        "soil_moisture": 100,
        "temperature": 30.0,
        "humidity": 70.0
    }

    response = client.post(
        "/sensor-readings",
        json=sensor_data
    )

    assert response.status_code == 200


def test_soil_moisture_above_upper_boundary():
    sensor_data = {
        "field_id": 1,
        "soil_moisture": 100.1,
        "temperature": 30.0,
        "humidity": 70.0
    }

    response = client.post(
        "/sensor-readings",
        json=sensor_data
    )

    assert response.status_code == 422


def test_irrigation_required():
    response = client.get("/irrigation-recommendation/25")

    assert response.status_code == 200
    assert response.json()["irrigation_required"] is True


def test_irrigation_monitoring():
    response = client.get("/irrigation-recommendation/45")

    assert response.status_code == 200
    assert response.json()["recommendation"] == "Monitor soil moisture"


def test_irrigation_not_required():
    response = client.get("/irrigation-recommendation/75")

    assert response.status_code == 200
    assert response.json()["recommendation"] == "No irrigation required"


def test_invalid_irrigation_value():
    response = client.get("/irrigation-recommendation/150")

    assert response.status_code == 400


def test_sensor_to_irrigation_recommendation():
    readings = client.get("/sensor-readings").json()
    reading_id = readings[0]["id"]

    response = client.get(
        f"/sensor-readings/{reading_id}/irrigation-recommendation"
    )

    assert response.status_code == 200
    assert response.json()["reading_id"] == reading_id
    assert "recommendation" in response.json()


def test_sensor_irrigation_missing_reading():
    response = client.get(
        "/sensor-readings/99999/irrigation-recommendation"
    )

    assert response.status_code == 404


def test_create_inventory_item():
    item = {
        "name": "Test Fertilizer",
        "category": "Fertilizer",
        "quantity": 10,
        "unit": "kg"
    }

    response = client.post("/inventory", json=item)

    assert response.status_code == 201
    assert response.json()["name"] == "Test Fertilizer"


def test_get_inventory():
    response = client.get("/inventory")

    assert response.status_code == 200
    assert isinstance(response.json(), list)
    assert len(response.json()) >= 1


def test_invalid_inventory_quantity():
    item = {
        "name": "Invalid Fertilizer",
        "category": "Fertilizer",
        "quantity": -5,
        "unit": "kg"
    }

    response = client.post("/inventory", json=item)

    assert response.status_code == 422
'''

with open("PFMS/tests/test_api.py", "w") as file:
    file.write(test_code)


# ------------------------------------------------------------
# Run the tests
# ------------------------------------------------------------

print("\nRunning Phase 2 test suite...\n")

result = subprocess.run(
    ["pytest", "PFMS/tests/test_api.py", "-v"],
    capture_output=True,
    text=True,
    env=env_vars # Pass the modified environment variables here too
)

print(result.stdout)

if result.stderr:
    print("\nWarnings/errors:")
    print(result.stderr)

Sample PFMS data added successfully.

API status: 200
API response: {'message': 'Precision Farming Management System API is running'}

Running Phase 2 test suite...

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.12.1, anyio-4.14.2, typeguard-4.6.0
collecting ... collected 21 items

PFMS/tests/test_api.py::test_root_endpoint PASSED                        [  4%]
PFMS/tests/test_api.py::test_create_field PASSED                         [  9%]
PFMS/tests/test_api.py::test_get_fields PASSED                           [ 14%]
PFMS/tests/test_api.py::test_get_nonexistent_field PASSED                [ 19%]
PFMS/tests/test_api.py::test_create_sensor_reading PASSED                [ 23%]
PFMS/tests/test_api.py::test_sensor_reading_invalid_field PASSED         [ 28%]
PFMS/tests/test_api.py::test_invalid_soil_moisture PASSED 

In [ ]:
phase 3= Software Engineering / Testing

In [50]:
# Phase 3: Software Engineering and Testing
# This phase expands the test suite, checks failure cases, and records the results.

import os
import sys
import subprocess
from datetime import datetime

# Make sure the PFMS package can be imported from Colab
project_root = "/content"

if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Make sure the test package is recognized correctly
open("PFMS/__init__.py", "a").close()
open("PFMS/tests/__init__.py", "a").close()

# ---------------------------------------------------------
# 1. Create the expanded pytest test suite
# ---------------------------------------------------------

test_code = r'''
from fastapi.testclient import TestClient
from PFMS.app.main import app

client = TestClient(app)


# Basic API checks
def test_root_endpoint():
    response = client.get("/")
    assert response.status_code == 200
    assert "message" in response.json()


# Field management tests
def test_create_field():
    response = client.post(
        "/fields",
        json={
            "name": "Phase 3 Test Field",
            "location": "Chennai",
            "crop_type": "Tomato"
        }
    )
    assert response.status_code == 201
    assert response.json()["name"] == "Phase 3 Test Field"


def test_get_fields():
    response = client.get("/fields")
    assert response.status_code == 200
    assert isinstance(response.json(), list)


def test_get_nonexistent_field():
    response = client.get("/fields/999999")
    assert response.status_code == 404


def test_create_field_with_missing_data():
    response = client.post(
        "/fields",
        json={
            "name": "Incomplete Field"
        }
    )
    assert response.status_code == 422


# Sensor reading tests
def test_create_sensor_reading():
    response = client.post(
        "/sensor-readings",
        json={
            "field_id": 1,
            "soil_moisture": 45,
            "temperature": 30,
            "humidity": 65
        }
    )
    assert response.status_code == 200
    assert response.json()["soil_moisture"] == 45


def test_sensor_reading_invalid_field():
    response = client.post(
        "/sensor-readings",
        json={
            "field_id": 999999,
            "soil_moisture": 45,
            "temperature": 30,
            "humidity": 65
        }
    )
    assert response.status_code == 404


def test_get_sensor_readings():
    response = client.get("/sensor-readings")
    assert response.status_code == 200
    assert isinstance(response.json(), list)


def test_get_nonexistent_sensor_reading():
    response = client.get("/sensor-readings/999999")
    assert response.status_code == 404


def test_update_sensor_reading():
    response = client.put(
        "/sensor-readings/1",
        json={
            "field_id": 1,
            "soil_moisture": 52,
            "temperature": 29,
            "humidity": 70
        }
    )
    assert response.status_code == 200
    assert response.json()["soil_moisture"] == 52


def test_update_nonexistent_sensor_reading():
    response = client.put(
        "/sensor-readings/999999",
        json={
            "field_id": 1,
            "soil_moisture": 52,
            "temperature": 29,
            "humidity": 70
        }
    )
    assert response.status_code == 404


def test_delete_nonexistent_sensor_reading():
    response = client.delete("/sensor-readings/999999")
    assert response.status_code == 404


# Soil moisture boundary and validation tests
def test_soil_moisture_lower_boundary():
    response = client.post(
        "/sensor-readings",
        json={
            "field_id": 1,
            "soil_moisture": 0,
            "temperature": 30,
            "humidity": 60
        }
    )
    assert response.status_code == 200


def test_soil_moisture_upper_boundary():
    response = client.post(
        "/sensor-readings",
        json={
            "field_id": 1,
            "soil_moisture": 100,
            "temperature": 30,
            "humidity": 60
        }
    )
    assert response.status_code == 200


def test_soil_moisture_above_upper_boundary():
    response = client.post(
        "/sensor-readings",
        json={
            "field_id": 1,
            "soil_moisture": 100.1,
            "temperature": 30,
            "humidity": 60
        }
    )
    assert response.status_code == 422


def test_soil_moisture_below_lower_boundary():
    response = client.post(
        "/sensor-readings",
        json={
            "field_id": 1,
            "soil_moisture": -0.1,
            "temperature": 30,
            "humidity": 60
        }
    )
    assert response.status_code == 422


def test_humidity_above_upper_boundary():
    response = client.post(
        "/sensor-readings",
        json={
            "field_id": 1,
            "soil_moisture": 50,
            "temperature": 30,
            "humidity": 100.1
        }
    )
    assert response.status_code == 422


def test_humidity_below_lower_boundary():
    response = client.post(
        "/sensor-readings",
        json={
            "field_id": 1,
            "soil_moisture": 50,
            "temperature": 30,
            "humidity": -0.1
        }
    )
    assert response.status_code == 422


def test_invalid_sensor_data_type():
    response = client.post(
        "/sensor-readings",
        json={
            "field_id": 1,
            "soil_moisture": "not-a-number",
            "temperature": 30,
            "humidity": 60
        }
    )
    assert response.status_code == 422


# Irrigation decision tests
def test_irrigation_required():
    response = client.get("/irrigation-recommendation/25")
    assert response.status_code == 200
    assert response.json()["irrigation_required"] is True


def test_irrigation_monitoring():
    response = client.get("/irrigation-recommendation/45")
    assert response.status_code == 200
    assert response.json()["irrigation_required"] is False
    assert response.json()["recommendation"] == "Monitor soil moisture"


def test_irrigation_not_required():
    response = client.get("/irrigation-recommendation/75")
    assert response.status_code == 200
    assert response.json()["irrigation_required"] is False
    assert response.json()["recommendation"] == "No irrigation required"


def test_irrigation_lower_boundary():
    response = client.get("/irrigation-recommendation/0")
    assert response.status_code == 200
    assert response.json()["irrigation_required"] is True


def test_irrigation_upper_boundary():
    response = client.get("/irrigation-recommendation/100")
    assert response.status_code == 200
    assert response.json()["irrigation_required"] is False


def test_invalid_irrigation_value():
    response = client.get("/irrigation-recommendation/101")
    assert response.status_code == 400


def test_negative_irrigation_value():
    response = client.get("/irrigation-recommendation/-1")
    assert response.status_code == 400


def test_sensor_to_irrigation_recommendation():
    response = client.get("/sensor-readings/1/irrigation-recommendation")
    assert response.status_code == 200
    assert "recommendation" in response.json()


def test_sensor_irrigation_missing_reading():
    response = client.get(
        "/sensor-readings/999999/irrigation-recommendation"
    )
    assert response.status_code == 404


# Inventory tests
def test_create_inventory_item():
    response = client.post(
        "/inventory",
        json={
            "name": "Phase 3 Test Fertilizer",
            "category": "Fertilizer",
            "quantity": 25,
            "unit": "kg"
        }
    )
    assert response.status_code == 201
    assert response.json()["quantity"] == 25


def test_get_inventory():
    response = client.get("/inventory")
    assert response.status_code == 200
    assert isinstance(response.json(), list)


def test_invalid_inventory_quantity():
    response = client.post(
        "/inventory",
        json={
            "name": "Invalid Fertilizer",
            "category": "Fertilizer",
            "quantity": -5,
            "unit": "kg"
        }
    )
    assert response.status_code == 422


def test_inventory_missing_name():
    response = client.post(
        "/inventory",
        json={
            "category": "Fertilizer",
            "quantity": 10,
            "unit": "kg"
        }
    )
    assert response.status_code == 422


# Test invalid HTTP input for an endpoint that expects a number
def test_invalid_field_id_format():
    response = client.get("/fields/not-a-number")
    assert response.status_code == 422


def test_invalid_sensor_id_format():
    response = client.get("/sensor-readings/not-a-number")
    assert response.status_code == 422
'''

with open("PFMS/tests/test_api.py", "w") as file:
    file.write(test_code)

print("Expanded Phase 3 pytest suite created.")


# ---------------------------------------------------------
# 2. Run the complete test suite
# ---------------------------------------------------------

print("\nRunning Phase 3 test suite...\n")

result = subprocess.run(
    ["pytest", "PFMS/tests/test_api.py", "-v"],
    capture_output=True,
    text=True
)

print(result.stdout)

if result.stderr:
    print(result.stderr)


# ---------------------------------------------------------
# 3. Record the test execution result
# ---------------------------------------------------------

passed = result.returncode == 0

if passed:
    test_status = "PASS"
    print("Phase 3 automated testing completed successfully.")
else:
    test_status = "FAIL"
    print("Some Phase 3 tests failed. Review the output above.")


# ---------------------------------------------------------
# 4. Save a simple test execution report
# ---------------------------------------------------------

report = f"""
PFMS - Phase 3 Test Execution Report
=====================================

Date: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

Testing Tool:
pytest

Application:
Precision Farming Management System

Test Scope:
- API availability
- Field management
- Sensor reading management
- Input validation
- Boundary value testing
- Negative testing
- Irrigation recommendation logic
- Inventory management
- Invalid input formats

Test Execution Status:
{test_status}

Return Code:
{result.returncode}

Testing Approach:
The test suite contains positive tests, negative tests, boundary value tests,
validation tests, and failure scenario tests.

Defect Status:
No confirmed application defect was identified if all tests pass.
Any failed test should be investigated before being classified as a defect.
"""

with open("PFMS/phase3_test_report.txt", "w") as file:
    file.write(report)

print("\nTest execution report saved to:")
print("PFMS/phase3_test_report.txt")


# ---------------------------------------------------------
# 5. Create a defect log for the assignment
# ---------------------------------------------------------

defect_log = """
PFMS - Phase 3 Defect Log
=========================

Defect ID | Description | Severity | Status
----------|-------------|----------|-------
DEF-001   | No confirmed defects found during the Phase 3 automated test execution. | - | Closed

Notes:
The defect log is maintained separately from the test results so that any
future failed test can be documented and tracked without changing the
original test evidence.
"""

with open("PFMS/defect_log.txt", "w") as file:
    file.write(defect_log)

print("Defect log saved to:")
print("PFMS/defect_log.txt")


# ---------------------------------------------------------
# 6. Show a short summary for the notebook
# ---------------------------------------------------------

print("\n" + "=" * 55)
print("PHASE 3 SUMMARY")
print("=" * 55)

if passed:
    print("✓ Pytest test suite executed successfully")
    print("✓ Positive test cases covered")
    print("✓ Negative test cases covered")
    print("✓ Boundary value cases covered")
    print("✓ Input validation cases covered")
    print("✓ Failure scenarios covered")
    print("✓ Test execution report created")
    print("✓ Defect log created")
else:
    print("✗ Test suite contains failures")
    print("Review the pytest output before documenting Phase 3 as complete.")

print("=" * 55)

Expanded Phase 3 pytest suite created.

Running Phase 3 test suite...

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: langsmith-0.12.1, anyio-4.14.2, typeguard-4.6.0
collecting ... collected 34 items

PFMS/tests/test_api.py::test_root_endpoint PASSED                        [  2%]
PFMS/tests/test_api.py::test_create_field PASSED                         [  5%]
PFMS/tests/test_api.py::test_get_fields PASSED                           [  8%]
PFMS/tests/test_api.py::test_get_nonexistent_field PASSED                [ 11%]
PFMS/tests/test_api.py::test_create_field_with_missing_data PASSED       [ 14%]
PFMS/tests/test_api.py::test_create_sensor_reading PASSED                [ 17%]
PFMS/tests/test_api.py::test_sensor_reading_invalid_field PASSED         [ 20%]
PFMS/tests/test_api.py::test_get_sensor_readings PASSED                  

In [52]:
# Phase 4: Create the final project documentation and configuration files

import os

project_root = "/content/PFMS"

# Make sure the project folder exists
os.makedirs(project_root, exist_ok=True)


# ---------------------------------------------------------
# 1. requirements.txt
# ---------------------------------------------------------

requirements = """fastapi
uvicorn
sqlalchemy
pydantic
pytest
httpx
"""

with open(f"{project_root}/requirements.txt", "w") as file:
    file.write(requirements)

print("requirements.txt created.")


# ---------------------------------------------------------
# 2. .gitignore
# ---------------------------------------------------------

gitignore = """# Python cache files
__pycache__/
*.py[cod]

# Virtual environments
vencv/
.venv/
env/

# Test and tool caches
.pytest_cache/
.coverage
htmlcov/

# SQLite database generated during local execution
*.db
*.sqlite
*.sqlite3

# Jupyter / Colab temporary files
.ipynb_checkpoints/

# Operating system files
.DS_Store
Thumbs.db

# IDE files
.vscode/
.idea/

# Environment variables and local configuration
.env
"""

with open(f"{project_root}/.gitignore", "w") as file:
    file.write(gitignore)

print(".gitignore created.")


# ---------------------------------------------------------
# 3. Professional README.md
# ---------------------------------------------------------

readme = """# Precision Farming Management System

A REST API for agricultural monitoring and resource management, developed using Python and FastAPI.

The system simulates agricultural field sensor data and provides basic decision support for irrigation management. It also manages agricultural resource inventory and demonstrates software testing practices using pytest.

## Project Overview

The Precision Farming Management System (PFMS) is designed as a lightweight backend system for monitoring agricultural fields.

The application provides APIs for:

- Managing agricultural fields
- Recording soil and environmental sensor readings
- Validating sensor data
- Generating irrigation recommendations
- Managing fertilizer and agricultural inventory
- Testing normal, boundary, invalid, and failure scenarios

The project was developed as a Software Engineering project with emphasis on REST API development and systematic software testing.

## Key Features

### Field Management
- Create agricultural fields
- Store field location and crop information
- Retrieve registered fields
- Handle requests for unavailable fields

### Sensor Monitoring
- Record soil moisture, temperature, and humidity
- Retrieve sensor readings
- Update existing readings
- Delete sensor readings
- Validate sensor input values
- Verify that sensor readings belong to valid fields

### Irrigation Decision Support

The system uses a simple rule-based irrigation model based on soil moisture:

| Soil Moisture | Recommendation |
|---------------|----------------|
| Below 30% | Irrigation required |
| 30% - 60% | Monitor soil moisture |
| Above 60% | No irrigation required |

This rule-based implementation provides a simple baseline that can later be extended with crop-specific thresholds or machine-learning models.

### Inventory Management
- Add agricultural resources
- Store fertilizer quantities and units
- Retrieve inventory information
- Validate inventory quantities and required fields

## Technology Stack

- **Python**
- **FastAPI**
- **SQLAlchemy**
- **SQLite**
- **Pydantic**
- **pytest**
- **HTTPX**

## Project Structure

```text
PFMS/
│
├── app/
│   ├── __init__.py
│   ├── main.py
│   ├── database.py
│   ├── models.py
│   ├── schemas.py
│   ├── crud.py
│   └── irrigation.py
│
├── tests/
│   ├── __init__.py
│   └── test_api.py
│
├── seed_data.py
├── phase3_test_report.txt
├── defect_log.txt
├── requirements.txt
├── .gitignore
└── README.md
```
"""

with open(f"{project_root}/README.md", "w") as file:
    file.write(readme)

print("README.md created.")

requirements.txt created.
.gitignore created.
README.md created.
